# 🚗 Predicción de precios de autos usados

Este proyecto tiene como objetivo construir y evaluar modelos de machine learning
para predecir el precio de autos usados a partir de sus características.

El enfoque incluye un modelo baseline, regresión lineal y un modelo de ensamble,
comparando su desempeño mediante la métrica RMSE.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


## 📌 Carga de datos

In [ ]:
df = pd.read_csv("/datasets/car_data.csv")
df.head()


## 🔍 Exploración inicial

In [ ]:
df.info()


## 🧹 Limpieza de datos

In [ ]:
# Eliminar duplicados
df = df.drop_duplicates()

# Rellenar valores faltantes
for col in df.columns:
    if df[col].dtype != 'object':
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum()


## 🔧 Ingeniería de características

In [ ]:
# Variables categóricas → one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

df_encoded.head()


## 🎯 Definición de variables

In [ ]:
X = df_encoded.drop("price", axis=1)
y = df_encoded["price"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=12345
)


## 📏 Modelo baseline

In [ ]:
baseline_pred = np.full(len(y_valid), y_train.median())
baseline_rmse = mean_squared_error(y_valid, baseline_pred, squared=False)

baseline_rmse


## 📐 Regresión lineal

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_valid)
lr_rmse = mean_squared_error(y_valid, lr_pred, squared=False)

lr_rmse


## 🌲 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=12345,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_valid)
rf_rmse = mean_squared_error(y_valid, rf_pred, squared=False)

rf_rmse


## 📊 Comparación de modelos

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest"],
    "RMSE": [baseline_rmse, lr_rmse, rf_rmse]
})

results


## 💾 Exportación de resultados (para dashboard)

In [ ]:
results.to_csv("results/model_comparison_rmse.csv", index=False)


## ✅ Conclusión

El modelo Random Forest obtuvo el mejor desempeño según la métrica RMSE,
superando tanto al modelo baseline como a la regresión lineal.

Este resultado indica que los modelos de ensamble capturan mejor las relaciones
no lineales presentes en los datos de autos usados.